# Carga da série histórica nacional (NEX-GDDP-CMIP6)

Parte 1 — setup: imports e conexões (S3 + Postgres).


In [1]:
import os 
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import xarray as xr
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from botocore.config import Config
import sys
sys.path.append("../scripts") # achar o .py

from recortar_brasil import recortar_brasil

In [2]:
load_dotenv()

BUCKET = "nex-gddp-cmip6"
BASE_PREFIX = "NEX-GDDP-CMIP6/"

client = boto3.client(
    "s3",
    config=Config(signature_version=UNSIGNED, connect_timeout=10, read_timeout=30, retries={"max_attempts": 2})
)
usuario = os.environ["DB_USER"]
senha = os.environ["DB_PASSWORD"]
host = os.environ.get("DB_HOST", "localhost")
porta = os.environ.get("DB_PORT", "5432")
nome_banco = os.environ["DB_NAME"]
engine = create_engine(f"postgresql+psycopg2://{usuario}:{senha}@{host}:{porta}/{nome_banco}")

# Corrige o caminho garantindo que a pasta exista
os.makedirs("../data/raw", exist_ok=True)
destino_tmp = "../data/raw/_tmp_ano.nc"

In [3]:
# teste de conexão
engine.connect()
print("conectado")

conectado


#### Funções de baixar ano e agregar para série nacional


In [ ]:

def baixar_ano(client, model, scenario, variable, ano, destino_tmp):
  for sufixo in ["_v2.0", "_v1.1", ""]:
    print("ok")
    key = f"{BASE_PREFIX}{model}/{scenario}/r1i1p1f1/{variable}/{variable}_day_{model}_{scenario}_r1i1p1f1_gn_{ano}{sufixo}.nc"
    try:
      client.download_file(BUCKET, key, destino_tmp)
      return key
    except Exception:
      continue
  raise FileNotFoundError(f"Nenhuma verão encontrada para {variable}/{ano}")

In [5]:
def agregar_nacional(destino_tmp, variable):
    ds = xr.open_dataset(destino_tmp)
    ds_brasil = recortar_brasil(ds)
    da = ds_brasil[variable]
    media_diaria = da.mean(dim=["lat", "lon"], skipna=True)
    df = media_diaria.to_dataframe().reset_index()
    df = df.rename(columns={variable: "valor"})
    df["variavel"] = variable
    return df[["time", "variavel", "valor"]]

### Loop de teste que baixa, agrega e carrega no banco


In [ ]:
model = "ACCESS-CM2"
scenario = "historical"
variable = "pr"
ano_inicio = 1950
ano_fim = 1954
tabela = "clima_diario_nacional"

for ano in range(ano_inicio, ano_fim + 1):
  print(f"Processando {variable} {ano}...")
  try:
    baixar_ano(client, model, scenario, variable, ano, destino_tmp)
    df = agregar_nacional(destino_tmp, variable)
    df["modelo"] = model
    df.to_sql(tabela, engine, if_exists="append", index=False)
  except Exception as e:
    print(f" Erro: {ano} - {e}")
  finally:
    if os.path.exists(destino_tmp):
      os.remove(destino_tmp)

print("Download concluido")

confirmando se funcionou, ja que o codigo não retornou as linhas por ano


In [ ]:
pd.read_sql("SELECT COUNT(*) FROM clima_diario_nacional", engine)

,count
0,1826


In [ ]:
pd.read_sql("SELECT * FROM clima_diario_nacional ORDER BY time LIMIT 5", engine)

,time,variavel,valor,modelo
0,1950-01-01 12:00:00,pr,0.000077,ACCESS-CM2
1,1950-01-02 12:00:00,pr,0.000079,ACCESS-CM2
2,1950-01-03 12:00:00,pr,0.000077,ACCESS-CM2
3,1950-01-04 12:00:00,pr,0.000083,ACCESS-CM2
4,1950-01-05 12:00:00,pr,0.000083,ACCESS-CM2


Variáveis `tas`, `tasmax` e `tasmin`


In [13]:
for variable in ["tas", "tasmax", "tasmin"]:
  for ano in range(ano_inicio, ano_fim + 1):
    print(f"Processando {variable} {ano}...")
    try:
      baixar_ano(client, model, scenario, variable, ano, destino_tmp)
      df = agregar_nacional(destino_tmp, variable)
      df["modelo"] = model
      df.to_sql(tabela, engine, if_exists="append", index=False)
      print(f"{len(df)} linhas carregadas para {ano}") #ver o progresso
    except Exception as e:
      print(f" Erro: {ano} - {e}")
    finally:
      if os.path.exists(destino_tmp):
        os.remove(destino_tmp)

print("Download concluido")

Processando tas 1950...
ok
365 linhas carregadas para 1950
Processando tas 1951...
ok
365 linhas carregadas para 1951
Processando tas 1952...
ok
366 linhas carregadas para 1952
Processando tas 1953...
ok
365 linhas carregadas para 1953
Processando tas 1954...
ok
365 linhas carregadas para 1954
Processando tasmax 1950...
ok
365 linhas carregadas para 1950
Processando tasmax 1951...
ok
365 linhas carregadas para 1951
Processando tasmax 1952...
ok
366 linhas carregadas para 1952
Processando tasmax 1953...
ok
365 linhas carregadas para 1953
Processando tasmax 1954...
ok
365 linhas carregadas para 1954
Processando tasmin 1950...
ok
365 linhas carregadas para 1950
Processando tasmin 1951...
ok
365 linhas carregadas para 1951
Processando tasmin 1952...
ok
366 linhas carregadas para 1952
Processando tasmin 1953...
ok
365 linhas carregadas para 1953
Processando tasmin 1954...
ok
365 linhas carregadas para 1954
Download concluido


Funcionou, mas pela demora talvez eu procure uma outra forma pra download (2h20m de execução)


In [14]:
pd.read_sql("SELECT variavel, COUNT(*) FROM clima_diario_nacional GROUP BY variavel", engine)

,variavel,count
0,tasmax,1826
1,pr,3287
2,tas,1826
3,tasmin,1826


In [15]:
pd.read_sql("""
    SELECT time, variavel, valor 
    FROM clima_diario_nacional 
    WHERE variavel IN ('tas', 'tasmax', 'tasmin') 
    ORDER BY time LIMIT 12
""", engine)

,time,variavel,valor
0,1950-01-01 12:00:00,tas,297.72230
1,1950-01-01 12:00:00,tasmin,292.14368
2,1950-01-01 12:00:00,tasmax,303.30084
3,1950-01-02 12:00:00,tas,298.04602
4,1950-01-02 12:00:00,tasmax,303.17044
5,1950-01-02 12:00:00,tasmin,292.92163
6,1950-01-03 12:00:00,tasmin,292.78040
7,1950-01-03 12:00:00,tas,297.85970
8,1950-01-03 12:00:00,tasmax,302.93910
9,1950-01-04 12:00:00,tas,297.98724
